In [ ]:
import pyspark.sql.functions as F

In [ ]:
df = spark.read.option("header", "true").csv("s3a://loicverdier/landing_zone/football_transfermarkt/transfers.csv")

In [ ]:
# Convertir la colonne 'transfer_fee' en entier
df = df.withColumn("transfer_fee", col("transfer_fee").cast("int"))
df = df.withColumn("market_value_in_eur", col("market_value_in_eur").cast("int"))

# Filtrer les lignes où 'transfer_fee' est supérieur à 0
df = df.filter(col("transfer_fee") > 0)

In [ ]:
# Calcul de la différence entre le prix d'achat et la valeur de marché
df = df.withColumn("price_difference", col("transfer_fee") - col("market_value_in_eur"))

# Regrouper par 'to_club_name' et calculer la moyenne pondérée
df_result = df.groupBy("to_club_name") \
    .agg(
        # Moyenne du surcout d'achat
        (spark_sum(col("price_difference")) / count("to_club_name")).cast("int").alias("surcout_moyen"),
        # Total du surcout d'achat
        (spark_sum(col("price_difference"))).alias("surcout_total"),
        # Compter le nombre de transferts par club
        count("to_club_name").alias("number_of_transfers")
    )

# Trier les résultats par 'market_value_in_eur' de la plus élevée à la moins élevée
df_sorted = df_result.orderBy(col("surcout_total").desc())

# Afficher les résultats
df_sorted.show()

In [ ]:
target_club = "Man Utd"

# Filtrer 'from_club_name' sur un club précis
df_filtered = df.filter(col("to_club_name") == target_club)

# Trier les résultats par 'transfer_fee' de la plus élevée à la moins élevée
df_sorted = df_filtered.orderBy(col("transfer_fee").desc())

# Afficher les premières lignes pour vérifier
df_sorted.show(50)

In [ ]:
# Extraire l'année de la colonne `transfer_date`
df = df.withColumn("transfer_year", F.year("transfer_date"))

# Calculer les ventes : somme de transfer_fee lorsque le club est dans from_club_name
sales = (
    df.groupBy("from_club_name", "transfer_year")
    .agg(F.sum("transfer_fee").alias("total_sales"))
    .withColumnRenamed("from_club_name", "club_name")
)

# Calculer les achats : somme de transfer_fee lorsque le club est dans to_club_name
purchases = (
    df.groupBy("to_club_name", "transfer_year")
    .agg(F.sum("transfer_fee").alias("total_purchases"))
    .withColumnRenamed("to_club_name", "club_name")
)

# Fusionner les résultats : jointure sur club_name et transfer_year
result = (
    sales.join(purchases, on=["club_name", "transfer_year"], how="outer")
    .fillna(0)  # Remplir les valeurs manquantes avec 0
    .orderBy("club_name", "transfer_year")
)

# Afficher les résultats
result.show()